# BANC v888 4-Layer Circuit Construction
- Constructs a 4-layer feed-forward circuit between OviDNs and CA neurons
- Uses `banc_888_edgelist_simple_v3.feather` and `banc_888_meta.feather`
- Identifies the finalized Layer 2 and Layer 3 neuron populations that connect to OviDN inputs to the CA neuron outputs
- This notebook was adapted from circuit construction code developed by Walker et al., 2025 (https://github.com/avdevineni/taste-connectome/blob/main/taste_connectome_analysis_Dec2024.ipynb)

### Part 1: Imports

In [180]:
# Part 1: Imports

import pandas as pd

### Part 2: Load Connectome Files and Input/Output Lists 
- current edgelist being used: "banc_888_edgelist_simple_v3.feather"
- current neuron-level metadata used: "banc_888_meta.feather"
- manually enter BANC v888 neuron IDs for desired input and output (OviDN input, CA neuron output in this case)

In [181]:
# Part 2: Settings and Neuron IDs

SYNAPSE_THRESHOLD = 3 # set minimum synapse threshold for edges (default = 3 for BANC connectome)

BANC_EDGE_FILE = "data/banc_888_edgelist_simple_v3.feather" # specify which BANC connection table to use
BANC_META_FILE = "data/banc_888_meta.feather" # specifiy which BANC neuron-level metadata file to use 

INPUT_NEURON_IDS = [ # currently contains OviDNs (n=16)
    720575941462378692,
    720575941596424581,
    720575941491649255,
    720575941524167755,
    720575941626745452,
    720575941607634285,
    720575941452030330,
    720575941561430829,
    720575941688560367,
    720575941460006128,
    720575941644058706,
    720575941521454003,
    720575941447935764,
    720575941529824440,
    720575941516601690,
    720575941417243164,
]

OUTPUT_NEURON_IDS = [ # currently contains CA neurons (n=6)
    720575941450146638,
    720575941490203081,
    720575941491151310,
    720575941506618428,
    720575941645177720,
    720575941655645652,
]

### Part 3: Load and Prepare BANC data
- clean the neuron IDs in the connections and meta tables
- proofread the neuron-level metadata file to only include neurons annotated as 'proofread' or 'roughly proofread'
- filter the connetions table to only include neurons the pass the proofread filter 

In [182]:
# Part 3: Load and Prepare BANC Connectivity

# Load and clean data 
connections_v3 = pd.read_feather(BANC_EDGE_FILE) # load the BANC connectivity data (BANC v888 v3 edgelist)
banc_meta = pd.read_feather(BANC_META_FILE) # load the BANC neuron-level metadata file 

connections_v3 = connections_v3.copy() # create a copy of the connections table before re-formatting 
connections_v3["pre"] = connections_v3["pre"].astype(str).str.strip() # clean the neuron IDs in the pre-synaptic column
connections_v3["post"] = connections_v3["post"].astype(str).str.strip() # clean the neuron IDs in the post-synaptic column
connections_v3["count"] = pd.to_numeric(connections_v3["count"], errors="raise") # ensure the synapse count values are numeric 

banc_meta = banc_meta.copy() # create a copy of the neuron-level metadata table before re-formatting 
banc_meta["banc_888_id"] = banc_meta["banc_888_id"].astype(str).str.strip() # clean the neuron IDs in the 'banc_888_id' column


# Proofread Graph Filter
proofread_mask = ( # identify neurons annotated as proofread or roughtly-proofread
    banc_meta["proofread"].astype(str).str.upper().eq("TRUE") # mark neurons as valid if they are proofread OR
    | banc_meta["roughly_proofread"].astype(str).str.upper().eq("TRUE") # mark neurs as valid if they are roughly_proofread
)

proofread_ids = set( # create a set containing all neuron IDs that pass the proofreading filter 
    banc_meta.loc[
        proofread_mask,
        "banc_888_id",
    ]
)

connections_v3 = connections_v3.loc[ # filter edges using proofread neurons 
    connections_v3["pre"].isin(proofread_ids) # make sure presynaptic neuron is proofread
    & connections_v3["post"].isin(proofread_ids) # make sure postsynaptic neuron is proofread 
].copy() # create a filtered copy of the edges table

connections = connections_v3.rename( #re-format and rename the columns in the connectivity file 
    columns={
        "pre": "pre_root_id",
        "post": "post_root_id",
        "count": "syn_count",
    }
)[["pre_root_id", "post_root_id", "syn_count"]].copy()

connections_agg = connections.copy() # create the final connectivity table used for circuit construction

print("Total connections after proofreading filter:", f"{len(connections_agg):,}")
print("Synapse threshold used:", SYNAPSE_THRESHOLD)

Total connections after proofreading filter: 13,370,448
Synapse threshold used: 3


### Part 4: Prepare Input and Output GRN lists
- clean and format dataframe containing input neuron lists (OviDNs)
- clean and format dataframe containing output neuron lists (CA neurons)

In [183]:
# Part 4: Prepare Input and Output GRN lists

input_neurons = pd.DataFrame({ # create a table of the manually-entered input neurons (OviDNs)
    "root_id": [str(x).strip() for x in INPUT_NEURON_IDS] # clean the neuron IDs in this table 
})

output_neurons = pd.DataFrame({ # create a table of the manually-entered output neurons (CA neurons)
    "root_id": [str(x).strip() for x in OUTPUT_NEURON_IDS] # clean the neuron IDs in this table 
})

print("Total input neurons:", len(input_neurons)) # print the total number of input neurons 
print("Total output neurons:", len(output_neurons)) # print the otal number of output neurons 

Total input neurons: 16
Total output neurons: 6


### Part 5: Find Layer 2 Candidates 
- identify connetions in which the pre-root ID is an input neuron (OviDNs)
- filter those connections by syanpse threshold
- from those connections, extract list of unique Layer 2 neurons as "Layer 2 Candidates"

In [184]:
# Part 5: Identify Layer 2 Candidates

# define helper function to identify neurons directly post-synaptic to Layer 1 inputs (OviDNs)
def find_Layer2_candidates(general_input_neurons, connection_df, synapse_threshold=SYNAPSE_THRESHOLD): # takes input neurons, connection table, and synapse threshold 
    Layer1_to_Layer2 = pd.merge( # create subset of connections
        general_input_neurons[["root_id"]], # look at the input neuron IDs 
        connection_df[["pre_root_id", "post_root_id", "syn_count"]], # look at these columns in the connections table 
        left_on="root_id", 
        right_on="pre_root_id" # only keep the connections in which an input neuron is the pre-root ID
    )

    Layer1_to_Layer2 = Layer1_to_Layer2[ # apply the minimum synapse threshold 
        Layer1_to_Layer2["syn_count"] >= synapse_threshold # only keep the connections that are greater than the synapse threshold 
    ]

    Layer1_to_Layer2 = Layer1_to_Layer2[ # only keep the pre-root ID, post-root ID, and the syn-count columns of the new connections subset
        ["pre_root_id", "post_root_id", "syn_count"]
    ]

    Layer1_to_Layer2 = Layer1_to_Layer2.reset_index(drop=True) # reset the index after filtering 

    return Layer1_to_Layer2 # output of this function will be a list of all candidate Input-Layer 2 connections 


Layer1_to_Layer2_candidates_connections = find_Layer2_candidates( # run the function we just built using: 
    input_neurons, # the list of input neurons 
    connections_agg # the filtered connection table 
)

Layer2_candidates = Layer1_to_Layer2_candidates_connections["post_root_id"].unique() # get the unique Layer 2 neuron IDs

print("Layer 2 candidates:", len(Layer2_candidates)) # print the total number of Layer 2 candidates 


Layer 2 candidates: 738


### Part 6: Find Layer 3 Candidates 
- identify connetions in which the post-root ID is an output neuron (CA neurons)
- filter those connections by syanpse threshold
- from those connections, extract list of unique Layer 3 neurons as "Layer 3 Candidates"
- exclude potential output neurons from Layer 3 candidates 
- exclude potential layer 2 neurons from Layer 3 candidates 
    - any neuron that is directly post-synaptic to input is assigned as Layer 2
    - even if this neuron directly conencts to output (2 hop), ensures the same neuron isn't in layer 2 and layer 3

In [185]:
# Part 6: Identify Layer 3 Candidates

# define a helper function that identifies neurons directly presynaptic to Layer 4 outputs (CA neurons)
def find_Layer3_candidates(general_output_neurons, connection_df, synapse_threshold=SYNAPSE_THRESHOLD): # takes output neurons, connection table, and synapse threshold 
    Layer3_to_Layer4 = pd.merge( # create subset of connections
        general_output_neurons[["root_id"]], # look at the output neuron IDs 
        connection_df[["pre_root_id", "post_root_id", "syn_count"]], # look at these columns in the connections table 
        left_on="root_id",
        right_on="post_root_id" # only keep the connections in which an output neuron is the post-root ID
    )

    Layer3_to_Layer4 = Layer3_to_Layer4[ # apply the minimum synapse threshold 
        Layer3_to_Layer4["syn_count"] >= synapse_threshold # only keep the connections that are greater than the synapse threshold 
    ]

    Layer3_to_Layer4 = Layer3_to_Layer4[ # only keep the pre-root ID, post-root ID, and the syn-count columns of the new connections subset
        ["pre_root_id", "post_root_id", "syn_count"]
    ]

    Layer3_to_Layer4 = Layer3_to_Layer4.reset_index(drop=True) # reset row numbers

    return Layer3_to_Layer4 # output of this function will be a list of all candidate Layer3-Output connections 


Layer3_to_Layer4_candidates_connections = find_Layer3_candidates( # run the function we just built using: 
    output_neurons, # the list of output neurons 
    connections_agg # the filtered connection table 
)

Layer3_to_Layer4_candidates_connections = Layer3_to_Layer4_candidates_connections[ # make sure Layer 3 candidates are not other output neurons
    ~Layer3_to_Layer4_candidates_connections["pre_root_id"].isin(output_neurons["root_id"]) # remove output neuron IDs from Layer 3 candidate connections
]

Layer3_to_Layer4_candidates_connections = Layer3_to_Layer4_candidates_connections[ # some neurons may be in Layer 2 and Layer 3 if they connect to both input and output directly (2-hop)
    ~Layer3_to_Layer4_candidates_connections["pre_root_id"].isin(Layer2_candidates) # make sure any Layer 2 neuron is only in Layer 2 and does not repeat in Layer 3
]

Layer3_candidates = Layer3_to_Layer4_candidates_connections["pre_root_id"].unique() # # get the unique Layer 2 neuron IDs
print("Layer 3 candidates:", len(Layer3_candidates)) # print the total number of Layer 2 candidates 

Layer 3 candidates: 59


### Part 7: Identify Layer 2 - Layer 3 Conenctions 
- find connections in which a layer 2 candidate is the pre-root ID and a layer 3 candidate is the post-root ID
- filter those connections based on synapse threshold  

In [186]:
# Part 7: Identify Layer 2 - Layer 3 Connections

L2_to_L3_connections = connections_agg[ # create subset of connections table that captures all L2-L3 connections
    (connections_agg["pre_root_id"].isin(Layer2_candidates)) # find connections that have a layer 2 candidate as a pre-root ID AND
    & (connections_agg["post_root_id"].isin(Layer3_candidates)) # connections that have a layer 3 candidate as a post-root ID
].copy() # create a copy of this connections subset

L2_to_L3_connections = L2_to_L3_connections[ 
    L2_to_L3_connections["syn_count"] >= SYNAPSE_THRESHOLD # keep connections that meet the minimum synapse threshold
].copy()

L2_to_L3_connections = L2_to_L3_connections.reset_index(drop=True) # reset row numbers

print("Number of L2 → L3 connections:", len(L2_to_L3_connections)) # print the total number of L2-L3 connections 


Number of L2 → L3 connections: 72


### Part 8: Identify potential Layer 2 - Layer 4 Connections 
- some Layer 2 neuron that are directly post-synaptic to input may directly connect to output (2 hop pathways)
- check if any such connections exist
- if so, return the Layer 2 candidate neuron(s) comprising the 2-hop pathway 

In [187]:
# Part 8: Identify any layer 2 neurons with direct connections to layer 4

L2_direct_to_output = connections_agg[ # create subset of connections table that captures all L2-output connections
    (connections_agg["pre_root_id"].isin(Layer2_candidates)) # find connections that have a layer 2 candidate as a pre-root ID AND
    & (connections_agg["post_root_id"].isin(output_neurons["root_id"])) # have a output neuron as a post-root ID AND
    & (connections_agg["syn_count"] >= SYNAPSE_THRESHOLD) # meet the minimum synapse threshold 
]

L2_direct_to_output_ids = L2_direct_to_output["pre_root_id"].unique().tolist() # get the unique Layer 2 neuron IDs, convert to list 

print("Layer 2 neurons directly connected to Layer 4:") 
print(L2_direct_to_output_ids) # print any Layer 2 neurons that connect to output directly 

Layer 2 neurons directly connected to Layer 4:
['720575941506055874']


### Part 9: Finalize and Export Layer 2 and Layer 3 Neurons 
- from the L2-L3 connections subset we built, exract the pre-root IDs as finalized Layer 2 neurons and the post-root IDs as the finalized Layer 3 neurons 
- export the dataframes containing these lists as csv files
    - Note: we externally added neurotransmitter identity data to these neuron lists after exporting

In [188]:
# Part 9: Finalize and Export Layer 2 and Layer 3 Neuron Lists

L2_final = pd.DataFrame({ # create a final Layer 2 neuron table
    "root_id": L2_to_L3_connections["pre_root_id"].unique() # identify unique Layer 2 neurons with connections to Layer 3
})

L3_final = pd.DataFrame({ # create the final Layer 3 neuron table
    "root_id": L2_to_L3_connections["post_root_id"].unique() # identify unique Layer 3 neurons receiving connections from Layer 2
})

L2_final = L2_final.reset_index(drop=True) # reset the Layer 2 table row numbers
L3_final = L3_final.reset_index(drop=True) # reset the Layer 3 table row numbers

print("Finalized Layer 2 neurons exported. Final Layer 2 count:", len(L2_final))
print("Finalized Layer 3 neurons exported. Final Layer 3 count:", len(L3_final))


# Export Final Neuron Lists

L2_final.to_csv("BANC_Layer2_neurons.csv", index=False) # export the final Layer 2 neuron list
L3_final.to_csv("BANC_Layer3_neurons.csv", index=False) # export the final Layer 3 neuron list


Finalized Layer 2 neurons exported. Final Layer 2 count: 37
Finalized Layer 3 neurons exported. Final Layer 3 count: 32


### Part 10: Plot Sankey

In [ ]:
import plotly.graph_objects as go

L1L2_mask = connections_agg['pre_root_id'].isin(input_neurons['root_id']) & connections_agg['post_root_id'].isin(L2_final['root_id'])
L2L3_mask = connections_agg['pre_root_id'].isin(L2_final['root_id']) & connections_agg['post_root_id'].isin(L3_final['root_id'])
L3L4_mask = connections_agg['pre_root_id'].isin(L3_final['root_id']) & connections_agg['post_root_id'].isin(output_neurons['root_id'])
L2L4_mask = connections_agg['pre_root_id'].isin(L2_final['root_id']) & connections_agg['post_root_id'].isin(output_neurons['root_id'])

sankey_edges = connections_agg[L1L2_mask | L2L3_mask | L3L4_mask | L2L4_mask].copy() # copy so edits don't touch connections_agg
sankey_edges = sankey_edges[sankey_edges['syn_count'] >= SYNAPSE_THRESHOLD]
print(f"{len(sankey_edges)} edges in the Sankey diagram.")

# collapse the input and output neurons into single nodes (only edit the ID column, not the whole row)
sankey_edges.loc[sankey_edges['pre_root_id'].isin(input_neurons['root_id']), 'pre_root_id'] = 'OviDNs'
sankey_edges.loc[sankey_edges['post_root_id'].isin(output_neurons['root_id']), 'post_root_id'] = 'CA neurons'

# bucket interneurons by neurotransmitter type
L2_nts = L2_final.copy().merge(banc_meta[['banc_888_id', 'neurotransmitter_predicted']], left_on='root_id', right_on='banc_888_id', how='left')
L2_nts = L2_nts.fillna("unknown")
L2_nt_counts = L2_nts.groupby('neurotransmitter_predicted').size().reset_index(name='count').sort_values('count', ascending=False)
print(L2_nt_counts.columns)

L3_nts = L3_final.copy().merge(banc_meta[['banc_888_id', 'neurotransmitter_predicted']], left_on='root_id', right_on='banc_888_id', how='left')
L3_nts = L3_nts.fillna("unknown")
L3_nt_counts = L3_nts.groupby('neurotransmitter_predicted').size().reset_index(name='count').sort_values('count', ascending=False)

def bucket_neuron(n):
    layer = None
    nt = None
    count = None

    if n in L2_final['root_id'].values:
        print("in L2")
        layer = "L2"
        nt = L2_nts[L2_nts['root_id'] == n].iloc[0]['neurotransmitter_predicted']
        print("got nt", nt)
        count = L2_nt_counts[L2_nt_counts['neurotransmitter_predicted'] == nt].iloc[0]['count']
        print("got count")
    elif n in L3_final['root_id'].values:
        print("in L3")
        layer = "L3"
        nt = L3_nts[L3_nts['root_id'] == n].iloc[0]['neurotransmitter_predicted']
        print("got nt", nt)
        count = L3_nt_counts[L3_nt_counts['neurotransmitter_predicted'] == nt]
        print(count)
        count = count.iloc[0]['count']
        print("got count")
    else:
        return n

    nt = 'GABA' if nt == 'gaba' else nt
    return f"{layer} {nt} (n={count})"

color_map = {
    "OviDNs": "#68B45C",
    "L2 dopamine (n=9)": "#B87969",
    "L2 GABA (n=17)": "#D5A848",
    "L2 serotonin (n=6)": "#8C6295",
    "L2 acetylcholine (n=2)": "#95A3CE",
    "L2 glutamate (n=3)": "#86A859",
    "L3 glutamate (n=10)": "#86A859",
    "L3 dopamine (n=9)": "#B87969",
    "L3 GABA (n=3)": "#D5A848",
    "L3 serotonin (n=3)": "#8C6295",
    "L3 acetylcholine (n=6)": "#95A3CE",
    "CA": "#8C9EDB",
}

sankey_edges['pre_root_id'] = sankey_edges['pre_root_id'].apply(bucket_neuron)
sankey_edges['post_root_id'] = sankey_edges['post_root_id'].apply(bucket_neuron)
print(sankey_edges)

# merge the edges that now share a source and target
sankey_edges = sankey_edges.groupby(['pre_root_id', 'post_root_id'], as_index=False)['syn_count'].sum()
print(f"{len(sankey_edges)} links after collapsing OviDNs and CA neurons.")

# plotly Sankey links refer to nodes by integer index into node.label
node_labels = pd.unique(pd.concat([sankey_edges['pre_root_id'], sankey_edges['post_root_id']])).tolist()
node_index = {label: i for i, label in enumerate(node_labels)}
source = sankey_edges['pre_root_id'].map(node_index).tolist()
target = sankey_edges['post_root_id'].map(node_index).tolist()
value = sankey_edges['syn_count'].tolist()
node_colors = [color_map.get(i, "grey") for i in node_index.keys()]
print(f"Nodes: {len(node_labels)}")

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color='black', width=0.5),
        label=node_labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
    )
)])

fig.update_layout(font_size=12, font_family='Arial')
fig.write_image('out/sankey.png', width=1200, height=900)

202 edges in the Sankey diagram.
Index(['neurotransmitter_predicted', 'count'], dtype='object')
in L2
got nt gaba
got count
in L2
got nt gaba
got count
in L2
got nt serotonin
got count
in L2
got nt gaba
got count
in L3
got nt acetylcholine
  neurotransmitter_predicted  count
0              acetylcholine      6
got count
in L3
got nt serotonin
  neurotransmitter_predicted  count
4                  serotonin      3
got count
in L2
got nt glutamate
got count
in L2
got nt glutamate
got count
in L3
got nt acetylcholine
  neurotransmitter_predicted  count
0              acetylcholine      6
got count
in L3
got nt acetylcholine
  neurotransmitter_predicted  count
0              acetylcholine      6
got count
in L3
got nt serotonin
  neurotransmitter_predicted  count
4                  serotonin      3
got count
in L2
got nt gaba
got count
in L3
got nt glutamate
  neurotransmitter_predicted  count
3                  glutamate     10
got count
in L3
got nt glutamate
  neurotransmitter_predicted